# 12 · S3 vs. HDFS Trade-offs (Case D)

**Theory**: docs/08-spark-and-object-storage.md

**Prerequisite**: this notebook is heavier than the others — it compares
Case C and Case D side by side, so it needs **both**
`make up-hadoop` and `make up-s3` running simultaneously. Close other
resource-hungry apps; this is the one moment in the lab where you're running
~10 containers at once.

In [ ]:
import sys
import time

sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, get_yarn_session, layer_path

yarn_spark = get_yarn_session("12-tradeoffs-yarn")
connect_spark = get_connect_session("12-tradeoffs-connect")

## Commit cost: rename-based (HDFS) vs. copy-based (S3)

HDFS implements `rename()` as an O(1) metadata pointer flip — Spark's commit
protocol (writing to a temp directory, then renaming to the final path)
is nearly free there. Object stores have no native rename: it's a full
**copy + delete**. We measure the same overwrite-write on both.

In [ ]:
vendas_local = layer_path("local", "bronze", "vendas")

start = time.perf_counter()
(yarn_spark.read.parquet("webhdfs://localhost:14000/datalake/bronze/vendas")
    .write.mode("overwrite")
    .parquet("webhdfs://localhost:14000/datalake/tmp/commit_test"))
hdfs_seconds = time.perf_counter() - start

start = time.perf_counter()
(connect_spark.read.parquet(layer_path("s3", "bronze", "vendas"))
    .write.mode("overwrite")
    .parquet("s3a://gold/commit_test"))
s3_seconds = time.perf_counter() - start

print(f"HDFS overwrite-write: {hdfs_seconds:.2f}s")
print(f"S3   overwrite-write: {s3_seconds:.2f}s")
print("\nThe gap widens with output size/partition count — HDFS's rename is")
print("O(1) regardless of file count; S3's commit is O(files) copy+delete.")

## Locality: a network trip either way, but different shapes

Neither Case C's executors (pulling from DataNodes over the Docker network)
nor Case D's (pulling from RustFS over the Docker network) get true "disk
-local" reads in this lab — Docker networking means everything crosses a
virtual network regardless. In a real bare-metal HDFS cluster, though,
executors would run **on the same physical machines** as the blocks they
read; S3 never offers that, by design (docs/08).

In [ ]:
yarn_spark.stop()
connect_spark.stop()